In [1]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.LoadupSamples import LoadupSamples
from src.predictionModule.FilterSamples import FilterSamples
from src.predictionModule.MachineModels import MachineModels

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import datetime
import optuna
import random
import torch
import copy

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import logging
formatted_date = datetime.datetime.now().strftime("%d%b%y_%H%M").lower()

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(fmt="%(asctime)s - %(message)s")
handler.setFormatter(formatter)
#if not logger.hasHandlers():
#    logger.addHandler(handler)
#else:
#    logger.handlers[:] = [handler]

#Output File handler
formatted_str = f"notebook-ydea-leavesLGB-{formatted_date}"
file_handler = logging.FileHandler(f"{formatted_str}.log", mode="w")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Usage
logger.setLevel(logging.INFO)
logger.info("This will print to the notebook's output cell")

c:\Users\KILightTouch\Desktop\RandomOdyssey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
params = {
    "idxAfterPrediction": 5,
    'timesteps': 90,
    'target_option': 'last',
    "LoadupSamples_tree_scaling_standard": True,
    "LoadupSamples_time_scaling_stretch": False,
    "LoadupSamples_time_inc_factor": 1,

    "FilterSamples_q_up": 0.6,
    
    "FilterSamples_cat_over20": True,
    "FilterSamples_cat_under2000": True,
    "FilterSamples_cat_posOneYearReturn": False,
    "FilterSamples_cat_posFiveYearReturn": False,

    "LSTM_val_split": 0.1,
}

In [4]:
timegroup = "group_regOHLCV_over5years"
treegroup = "group_debug"

logger.info(f"Using treegroup: {treegroup}, timegroup: {timegroup}")

eval_date = datetime.date(year=2025, month=7, day=13)
evaldates = [eval_date - datetime.timedelta(days=i) for i in range(1, 6)]
start_train_date = datetime.date(year=2016, month=1, day=1)
split_Date = datetime.date(year=2025, month=7, day=1)
ls = LoadupSamples(
    train_start_date=start_train_date,
    test_dates=evaldates,
    treegroup=treegroup,
    timegroup=timegroup,
    params=params,
)
ls.load_samples(main_path = "../src/featureAlchemy/bin/")
ls.split_dataset(
    start_date=start_train_date,
    last_train_date=split_Date,
    last_test_date=eval_date
)
fs_pre = FilterSamples(
    Xtree_train = ls.train_Xtree, 
    ytree_train = ls.train_ytree, 
    treenames   = ls.featureTreeNames,
    Xtree_test  = ls.test_Xtree,  
    ytree_test  = ls.test_ytree,
    meta_train  = ls.meta_pl_train, 
    meta_test   = ls.meta_pl_test, 
    params      = params
)
mask_train_pre, mask_test_pre = fs_pre.categorical_masks()
ls.apply_masks(mask_train_pre, mask_test_pre)

In [5]:
Xtree_train = ls.train_Xtree
ytree_train = ls.train_ytree
Xtree_test  = ls.test_Xtree
ytree_test  = ls.test_ytree

Xtime_train = ls.train_Xtime
ytime_train = ls.train_ytime
Xtime_test  = ls.test_Xtime
ytime_test  = ls.test_ytime

treenames   = ls.featureTreeNames
timenames   = ls.featureTimeNames
meta_train  = ls.meta_pl_train
meta_test   = ls.meta_pl_test

dates_tr = meta_train['date'].unique().sort()
dates_te = meta_test['date'].unique().sort()

from src.common.DataFrameTimeOperations import DataFrameTimeOperations as dfta
dates_tr_idx = dfta(meta_train, 'date').getNextLowerOrEqualIndices(dates_tr)
dates_te_idx = dfta(meta_test, 'date').getNextLowerOrEqualIndices(dates_te)

assert not any([i == -1 for i in dates_tr_idx])
assert not any([i == -1 for i in dates_te_idx])

nS, nT, nF = Xtime_train.shape

In [6]:
def geometric_mean_safe(arr):
    arr = np.asarray(arr, dtype=float)
    minv = np.min(arr) if arr.size else 0.0
    shift = -minv + 1e-9 if minv <= 0 else 0.0
    return float(np.exp(np.mean(np.log(arr + shift)))) if arr.size else np.nan

def metric(arr):
    """Custom cluster score function."""
    gm = geometric_mean_safe(arr)
    return gm - 1

def make_design(X, t_win):
    Xw: np.ndarray = X[:, -(t_win+1):, 0]
    eps = 1e-9
    inc_factor = params["LoadupSamples_time_inc_factor"]
    Xw_mid = np.clip((Xw-0.5)*2.0, -1.0+eps, 1.0-eps)
    Xw = np.arctanh(Xw_mid)/inc_factor + 1.0
    prev = Xw[:, :-1]
    curr = Xw[:, 1:]
    Xw = (curr / (prev + eps)) - 1.0
    Xw = np.clip(Xw, 1e-5, 1e5)
    
    return Xw.reshape(Xw.shape[0], -1)

In [7]:
Xd_tr = Xtree_train
Xd_te = Xtree_test

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def clean_train_test(
    Xtr_tree: np.ndarray,
    Xte_tree: np.ndarray,
    var_thresh: float = 1e-12,
    winsor_z: float | None = 6.0,
    scale: bool = True,
):
    """
    Pre-clean Xtr_tree/Xte_tree consistently:
      1) replace non-finite with NaN
      2) median-impute (fit on train)
      3) drop near-zero variance cols (by train variance)
      4) optional winsorize by train z-score (clip to ±winsor_z)
      5) optional standardize (fit on train)

    Returns:
      Xtr_clean, Xte_clean, keep_mask  (keep_mask maps kept feature columns)
    """
    Xtr = np.asarray(Xtr_tree, dtype=np.float64)
    Xte = np.asarray(Xte_tree, dtype=np.float64)

    # 1) finite
    Xtr = Xtr.copy(); Xte = Xte.copy()
    Xtr[~np.isfinite(Xtr)] = np.nan
    Xte[~np.isfinite(Xte)] = np.nan

    # 2) impute (fit on train)
    imp = SimpleImputer(strategy="median")
    Xtr = imp.fit_transform(Xtr)
    Xte = imp.transform(Xte)

    # 3) variance threshold (on train)
    var = Xtr.var(axis=0)
    keep_mask = var > var_thresh
    if not np.any(keep_mask):
        # fallback: keep at least one column
        keep_mask = var == var.max()
    Xtr = Xtr[:, keep_mask]
    Xte = Xte[:, keep_mask]

    # 4) winsorize (clip by train z-scores)
    if winsor_z is not None:
        mu = Xtr.mean(axis=0)
        sd = Xtr.std(axis=0)
        sd[sd == 0] = 1.0
        def _winsor(X):
            Z = (X - mu) / sd
            Z = np.clip(Z, -winsor_z, winsor_z)
            return Z * sd + mu
        Xtr = _winsor(Xtr)
        Xte = _winsor(Xte)

    # 5) scale (fit on train)
    if scale:
        sc = StandardScaler(with_mean=True, with_std=True)
        Xtr = sc.fit_transform(Xtr)
        Xte = sc.transform(Xte)

    return Xtr, Xte, keep_mask

In [ ]:
def __top_leaf_labels_per_tree(
    model: lgb.Booster,
    X,
    y,
    tree_n_max: int,
    top_n_max: int,
) -> tuple[np.ndarray, np.ndarray]:
    leaf_mat = model.predict(X, pred_leaf=True)  # shape: (n_samples, n_trees_total)
    if leaf_mat.ndim == 1:
        leaf_mat = leaf_mat.reshape(-1, 1) 
    logger.info(f"  leaf_mat: {leaf_mat.shape}.")
    n_trees_total = leaf_mat.shape[1]
    n_trees = min(tree_n_max, n_trees_total)

    y_arr = np.asarray(y, dtype=float)
    out_label = np.full((top_n_max, n_trees), -1, dtype=np.int32)
    out_score = np.full((top_n_max, n_trees), metric(1.0), dtype=float)

    for t in range(n_trees):
        labels_t = leaf_mat[:, t].astype(np.int32)

        df = pl.DataFrame({"label": labels_t, "y": y_arr})
        gb = df.group_by("label").agg([
            pl.count("y").alias("count_y"),
            pl.mean("y").alias("mean_y"),
            #pl.col("y").log().mean().exp().alias("gmean_y"),
            #pl.col("y").log().mean().alias("mean_logy"),
            #pl.col("y").log().std().alias("std_logy"),
            #pl.sum("y").alias("sum_y"),
            #pl.var("y").alias("var_y"),
            pl.std("y").alias("std_y"),
            #pl.max("y").alias("max_y"),
            #pl.min("y").alias("min_y"),
            #pl.median("y").alias("median_y"),
            #pl.quantile("y", 0.1).alias("q10_y"),
            #pl.quantile("y", 0.25).alias("q25_y"),
            #pl.quantile("y", 0.75).alias("q75_y"),
            #pl.quantile("y", 0.9).alias("q90_y"),
        ]).with_columns([
            (pl.col("mean_y") - 1.96 * pl.col("std_y") / pl.col("count_y").sqrt())
                .alias("lower_gmean")
        ]).with_columns([
            (pl.col("lower_gmean") - 1.0).alias("score")
        ])

        sorted_gb = (
            gb.with_columns(pl.col("score").fill_nan(0.0).fill_null(0.0))
            .sort(["score", "count_y"], descending=[True, True])
            .select(["label", "score"])
        )        
        arr = sorted_gb.to_numpy()  # shape (n_labels, 2)
        k = min(top_n_max, arr.shape[0])
        if k > 0:
            out_label[:k, t] = arr[:k, 0].astype(np.int32)
            out_score[:k, t] = arr[:k, 1].astype(float)

    return out_label, out_score

def pca_min_k(Xtr, Xte, pca_var=0.8, min_k=3, svd_solver="full"):
    # 1) fit once to get variance curve
    p_full = PCA(n_components=None, svd_solver=svd_solver)
    p_full.fit(Xtr)

    # how many to reach pca_var
    csum = np.cumsum(p_full.explained_variance_ratio_)
    k_var = int(np.searchsorted(csum, pca_var) + 1)

    # cap by matrix rank-ish limit
    k_cap = min(Xtr.shape[0], Xtr.shape[1])
    k = max(min_k, k_var)
    k = min(k, k_cap)

    # 2) refit with chosen k
    p = PCA(n_components=k, svd_solver=svd_solver)
    Xtr_pca = p.fit_transform(Xtr)
    Xte_pca = p.transform(Xte)
    return Xtr_pca, Xte_pca, p, k

def _score_once_PCA(
    Xtr_tree: np.ndarray,
    ytr_tree: np.ndarray,
    Xte_tree: np.ndarray,
    yte_tree: np.ndarray,
    mm: MachineModels,
    tree_n_max: int = 1,
    top_n_max: int = 1,
    min_n_tar: int = 0,
    pca_var = 0.8,
) -> float:
    
    Xtr_clean, Xte_clean, keep_mask = clean_train_test(Xtr_tree, Xte_tree)
    Xtr_pca, Xte_pca, pca, k = pca_min_k(Xtr_clean, Xte_clean, pca_var=pca_var, min_k=3)


    logger.info(f"  PCA reduced tree features from (TRAIN) {Xtr_tree.shape[1]} to {Xtr_pca.shape[1]} dimensions.")
    logger.info(f"  PCA reduced tree features from (TEST) {Xte_tree.shape[1]} to {Xte_pca.shape[1]} dimensions.")

    if Xte_pca.shape[0] < 2:
        return metric(1.0)

    try:
        logger.disabled = True
        model_lgb, info = mm.run_LGB(
            X_train=Xtr_pca,
            y_train=ytr_tree,
            X_test=Xte_pca,
            y_test=yte_tree,
        )
    except Exception as e:
        logger.disabled = False
        logger.warning(f"  LGB failed: {e}")
        return metric(1.0)
    finally:
        logger.disabled = False

    #bi = info.get("best_iteration", 1)
    tree_n_max = min(model_lgb.num_trees(), tree_n_max)
    labels_top, scores_top = __top_leaf_labels_per_tree(
        model_lgb, Xtr_pca, ytr_tree, tree_n_max=tree_n_max, top_n_max=max(1, top_n_max or 1)
    )
    
    # Top labels by score
    n_trees = len(scores_top[0,:])
    leaf_te = model_lgb.predict(Xte_pca, pred_leaf=True)[:, :n_trees]
    if leaf_te.ndim == 1:
        leaf_te = leaf_te.reshape(-1, 1) 
    # If everything is -1 across all ranks, bail out
    if labels_top.size == 0 or np.all(labels_top == -1):
        logger.warning("  LGB failed to generate predictions.")
        return metric(1.0)
    
    # Rank every (rank, tree) pair by descending score
    r_idx, t_idx = np.unravel_index(np.argsort(scores_top.ravel())[::-1], scores_top.shape)
    
    sel_pairs = []  # (tree_idx, rank_idx)
    mask_sel = np.zeros(leaf_te.shape[0], dtype=bool)
    y_selected = np.array([], dtype=yte_tree.dtype)
        
    for r, t in zip(r_idx, t_idx):
        lbl = labels_top[r, t]
        if lbl == -1:
            continue
        mask_sel |= (leaf_te[:, t] == lbl)
        sel_pairs.append((int(t), int(r)))
        y_selected = yte_tree[mask_sel]
        y_selected = y_selected[np.isfinite(y_selected)]
        if y_selected.size >= min_n_tar:
            break

    logger.info(
        f"  LGB selected {y_selected.size} out of {yte_tree.size} samples "
        f"using {len(sel_pairs)} (tree,rank) pairs: {sel_pairs}"
    )

    if y_selected.size == 0:
        logger.warning("  LGB failed to select testing values.")
        return metric(1.0)

    score = metric(y_selected)
    logger.info(
        f"LGB score={score:.6f}, selected={y_selected.size}/{yte_tree.size}"
    )

    return float(score) if np.isfinite(score) else metric(1.0)

In [47]:
# ---- knobs (use existing globals if present) ----
device = "cuda" if torch.cuda.is_available() else "cpu"
n_splits = 75
n_test_days = 7

max_training_days = 1301
N = len(dates_tr_idx)
lo = max_training_days - 1                      # min pivot (last train index)
hi = N - n_test_days - 2                      # max pivot
eligible = list(range(lo, hi + 1))
assert len(eligible) >= n_splits, f"Too few eligible pivots ({len(eligible)}) for n_splits={n_splits}"
pivots = sorted(random.sample(eligible, n_splits))

for p in pivots:
    logger.info(f"  Pivot {p}: Date {dates_tr[p]}")

In [ ]:
def objective() -> float:
    opt_params = copy.deepcopy(params)
    opt_params["LGB_num_boost_round"]           = 50
    opt_params["LGB_lambda_l1"]                 = 1e-1
    opt_params["LGB_lambda_l2"]                 = 1e-1
    opt_params["LGB_feature_fraction"]          = 1.0
    opt_params["LGB_num_leaves"]                = 100
    opt_params["LGB_max_depth"]                 = 13
    opt_params["LGB_learning_rate"]             = 0.3455661043652934
    opt_params["LGB_min_data_in_leaf"]          = 400
    opt_params["LGB_min_gain_to_split"]         = 0.00017474038986917118
    opt_params["LGB_path_smooth"]               = 0.011821683694174146
    opt_params["LGB_min_sum_hessian_in_leaf"]   = 0.04470523518057471
    opt_params["LGB_max_bin"]                   = 2000
    opt_params["LGB_early_stopping_rounds"]     = 35
    n_training_days =   opt_params["n_training_days"]   = 1000
    min_n_tar =         opt_params["min_n_tar"]         = 5
    top_n_max =         opt_params["top_n_max"]         = 6
    logger.info(f"Params: {opt_params}")
    scores = []
    mm = MachineModels(params=opt_params)
    for i in range(n_splits):
        p = pivots[i]
        tr_l_idx = dates_tr_idx[p - n_training_days + 1]
        tr_u_idx = dates_tr_idx[p + 1] - 1
        te_l_idx = dates_tr_idx[p + 1]
        te_u_idx = dates_tr_idx[p + n_test_days + 1] - 1
        # example: get date bounds if needed
        s_tr = slice(tr_l_idx, tr_u_idx+1)
        s_te = slice(te_l_idx, te_u_idx+1)
        Xtr, ytr_tree = Xd_tr[s_tr].copy(), ytree_train[s_tr].copy()
        Xte, yte_tree = Xd_tr[s_te].copy(), ytree_train[s_te].copy()
        
        #try:
        sc = _score_once_PCA(Xtr, ytr_tree, Xte, yte_tree, mm, 
                tree_n_max=opt_params["LGB_num_boost_round"]-1, top_n_max=top_n_max, min_n_tar=min_n_tar)
        #except Exception as e:
        #    logger.info(f"Exception during scoring: {e}")
        #    trial.should_prune()
        #    sc = metric(1.0)
        scores.append(sc)
    vals = [v for v in scores if np.isfinite(v)]
    logger.info(f"Scores per splits: {vals}")
    vals = np.array(vals)
    vals_log = np.log(1.0 + vals)
    if len(vals) < (len(scores)//2):
        return 0.0
    return float(np.mean(vals_log)) if len(vals_log) else -np.inf

objective()